# Random Forest — Re-test on Fresh Random Days

The original 9 test days were **put back into training**. A completely new set of **9 randomly selected days** (3 LOW / 3 MID / 3 HIGH) is used as the test set.

Random seed = 99 so the selection is reproducible but was not hand-picked.

**New test days:**

| Band | Date | Actual Wh | Temp C |
|---|---|---|---|
| HIGH | 2013-12-01 | 38,999 | 6.1 |
| HIGH | 2014-01-05 | 41,688 | 4.0 |
| LOW  | 2014-07-07 | 11,561 | 15.1 |
| LOW  | 2014-08-21 | 10,982 | 12.6 |
| LOW  | 2014-12-05 | 7,034  | 3.6 |
| MID  | 2015-01-24 | 15,895 | 3.6 |
| HIGH | 2015-03-26 | 40,805 | 5.8 |
| MID  | 2015-05-27 | 14,838 | 11.1 |
| MID  | 2015-07-01 | 15,767 | 24.8 |

In [ ]:
!pip install -q xgboost

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
print('Imports OK')

## 1. Upload Data
Upload `train.csv` and `test.csv` from `data/processed/` — these already have temperature columns merged in.

In [ ]:
from google.colab import files
print('Upload train.csv and test.csv')
uploaded = files.upload()

In [ ]:
train = pd.read_csv('train.csv', parse_dates=['datetime'])
test  = pd.read_csv('test.csv',  parse_dates=['datetime'])
print(f'Train: {len(train)} rows  |  Test: {len(test)} rows')

## 2. Prepare Data

In [ ]:
FEATURES = [
    'day_of_week', 'month', 'is_weekend',
    'lag_1', 'lag_7', 'rolling_mean_7',
    'heater_lag_1', 'heater_lag_7', 'heater_rolling_mean_7',
    'temp_mean_c', 'temp_min_c',
]

# combine train and test into one clean dataset
all_data = (
    pd.concat([train, test])
      .sort_values('datetime')
      .reset_index(drop=True)
)

# remove sensor-gap zeros, partial last day, days adjacent to gap
all_data = all_data[
    (all_data['aggregate_wh'] > 5000) &
    (all_data['datetime'] != '2015-07-10') &
    (all_data['lag_1'] > 5000) &
    (all_data['lag_7'] > 5000)
].dropna(subset=FEATURES + ['aggregate_wh']).reset_index(drop=True)

print(f'Clean rows: {len(all_data)}')
print(f'Date range: {all_data["datetime"].min().date()} → {all_data["datetime"].max().date()}')

## 3. Define New Test Days
9 randomly selected days (seed=99). The original 9 from the first experiment are **back in training**.

In [ ]:
# original 9 are now part of training — not excluded
NEW_TEST_DATES = pd.to_datetime([
    '2013-12-01',   # HIGH
    '2014-01-05',   # HIGH
    '2014-07-07',   # LOW
    '2014-08-21',   # LOW
    '2014-12-05',   # LOW
    '2015-01-24',   # MID
    '2015-03-26',   # HIGH
    '2015-05-27',   # MID
    '2015-07-01',   # MID
])

test_df  = all_data[all_data['datetime'].isin(NEW_TEST_DATES)].copy().reset_index(drop=True)
train_df = all_data[~all_data['datetime'].isin(NEW_TEST_DATES)].copy().reset_index(drop=True)

def band(wh):
    return 'LOW' if wh < 12000 else ('MID' if wh < 25000 else 'HIGH')

test_df['band'] = test_df['aggregate_wh'].apply(band)

show = test_df[['band','datetime','temp_mean_c','aggregate_wh']].copy()
show.columns = ['Band','Date','Temp C','Actual Wh']
show['Date']      = show['Date'].dt.strftime('%Y-%m-%d')
show['Actual Wh'] = show['Actual Wh'].round(0).astype(int)
show['Temp C']    = show['Temp C'].round(1)
print(show.sort_values('Band').to_string(index=False))
print(f'\nTraining rows: {len(train_df)}  |  Test rows: {len(test_df)}')

## 4. Train Random Forest

In [ ]:
X_train = train_df[FEATURES].values
y_train = train_df['aggregate_wh'].values
X_test  = test_df[FEATURES].values
y_test  = test_df['aggregate_wh'].values

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print('Training complete.')
print(f'Features used: {FEATURES}')

## 5. Results

In [ ]:
preds = model.predict(X_test)

results = []
for i, row in test_df.reset_index(drop=True).iterrows():
    actual = row['aggregate_wh']
    pred   = preds[i]
    results.append({
        'Band'        : row['band'],
        'Date'        : row['datetime'].strftime('%Y-%m-%d'),
        'Temp C'      : round(row['temp_mean_c'], 1),
        'Actual Wh'   : round(actual),
        'Predicted Wh': round(pred),
        'Error %'     : round(abs(pred - actual) / actual * 100, 1),
    })

df = pd.DataFrame(results).sort_values('Band').reset_index(drop=True)

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mape = np.mean(np.abs((y_test - preds) / y_test)) * 100

print('='*68)
print('  Random Forest — Fresh Random Test Set')
print('='*68)
print(df.to_string(index=False))
print()
print(f'  MAE  : {mae:,.0f} Wh')
print(f'  RMSE : {rmse:,.0f} Wh')
print(f'  MAPE : {mape:.1f}%')
print()
print(f'  First run (original 9 days) MAPE : 19.2%')
print(f'  This run  (random 9 days)   MAPE : {mape:.1f}%')
if mape <= 25:
    print('  Verdict: consistent — result holds on unseen data')
elif mape <= 40:
    print('  Verdict: moderate — model is less reliable on random days')
else:
    print('  Verdict: inconsistent — first result was optimistic')

## 6. Feature Importance

In [ ]:
importance = pd.DataFrame({
    'Feature'   : FEATURES,
    'Importance': model.feature_importances_,
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('Feature importance (higher = model relies on it more):')
print(importance.to_string(index=False))